In [11]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df=pd.read_csv("Dataset-Coginifiyz.csv",encoding='utf-8-sig')
df.head(1)

,ï»¿Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365


In [1]:
df.columns = df.columns.str.replace("ï»¿", "", regex=False).str.strip()

NameError: name 'df' is not defined

In [3]:
df.shape

(9551, 21)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ï»¿Restaurant ID      9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [5]:
df.isnull().sum()

ï»¿Restaurant ID        0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [7]:
r=df[['Restaurant Name','City','Cuisines','Price range','Average Cost for two','Aggregate rating']].copy()
r['Cuisines']=r['Cuisines'].fillna("")
r=r.drop_duplicates(subset=['Restaurant Name','City']).reset_index(drop=True)
r.shape

(7974, 6)

In [8]:
def recommend(cuisine,price_range=None,city=None,top_n=5):
    pool=r.copy()
    if city:
        pool=pool[pool['City'].str.lower()==city.lower()]
    if price_range:
        pool=pool[pool["Price range"]==price_range]
    if pool.empty:
        return "Not Found"

    tfidf=TfidfVectorizer(
        tokenizer=lambda s:[x.strip().lower() for x in s.split(",")],
        token_pattern=None
    )
    mat=tfidf.fit_transform(pool['Cuisines'])
    q=tfidf.transform([cuisine])
    pool=pool.assign(score=cosine_similarity(q,mat).ravel())

    return (pool.sort_values(['score','Aggregate rating'],ascending=False).head(top_n)[["Restaurant Name",'City','Cuisines','Price range','Aggregate rating','score']])

In [9]:
recommend("North indian, Chinese",price_range=2,city='Noida')

,Restaurant Name,City,Cuisines,Price range,Aggregate rating,score
6869,The Saffron Boutique,Noida,"North Indian, Chinese",2,4.0,1.0
7201,Platefull,Noida,"North Indian, Chinese",2,3.4,1.0
7377,World Bites,Noida,"Chinese, North Indian",2,3.4,1.0
7431,Chilli Zone,Noida,"North Indian, Chinese",2,3.4,1.0
6815,Punjabi Pakwaan,Noida,"North Indian, Chinese",2,3.3,1.0


In [10]:
df['City'].value_counts()

City
New Delhi           5473
Gurgaon             1118
Noida               1080
Faridabad            251
Ghaziabad             25
                    ... 
Panchkula              1
Mc Millan              1
Mayfield               1
Macedon                1
Vineland Station       1
Name: count, Length: 141, dtype: int64

In [18]:
recommend("South indian veg, ",price_range=2,city='Ghaziabad')

,Restaurant Name,City,Cuisines,Price range,Aggregate rating,score
1090,Mr. Brown,Ghaziabad,"Fast Food, Cafe, Desserts, Bakery",2,3.9,0.0
1107,Dunkin' Donuts,Ghaziabad,"Burger, Desserts, Fast Food",2,3.6,0.0
1108,Haldiram's,Ghaziabad,"North Indian, South Indian, Chinese, Street Fo...",2,3.6,0.0
1094,Barista,Ghaziabad,Cafe,2,3.4,0.0
1096,Burger King,Ghaziabad,"Burger, Fast Food",2,3.4,0.0
